In [32]:
import pandas as pd
import numpy as np
from datetime import timedelta
from tqdm import tqdm

# Загрузка кластеров, измерений, событий
clusters_df = pd.read_csv('data/clinical_report_clean.csv')
measurements = pd.read_csv('data/2025_04_15/Первичные данные от 10.04.2025.csv', 
                           usecols=['id пациента', 'время измерения', 'САД', 'ДАД', 'ЧП'])
kzs = pd.read_csv('data/2025_04_15/Клинически значимые события от 15.04.2025.csv')

In [33]:
import os

CACHE_DIR = 'cache'
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"✅ Папка для кешей: {CACHE_DIR}")

✅ Папка для кешей: cache


In [34]:
import pandas as pd
import numpy as np
from datetime import timedelta
from tqdm import tqdm # для визуализации процесса

def create_training_dataset(df_measurements, df_kzs, df_clusters, window_size=14, horizon=7):
    # 1. Приведение типов и подготовка
    df_measurements['timestamp'] = pd.to_datetime(df_measurements['время измерения'])
    df_measurements['date'] = df_measurements['timestamp'].dt.date
    
    df_kzs['kzs_date'] = pd.to_datetime(df_kzs['дата, время формирования КЗС']).dt.date
    
    # Коды критических событий (кризы)
    critical_codes = [33901, 33900, 33012, 33601, 33801]
    df_kzs_critical = df_kzs[df_kzs['код КЗС'].isin(critical_codes)]
    
    samples = []
    
    # Берем только тех пациентов, по которым у нас есть кластеры (уже очищенные)
    target_patients = df_clusters['patient_id'].unique()
    
    print(f"Начинаем нарезку окон для {len(target_patients)} пациентов...")
    
    for p_id in tqdm(target_patients):
        # Данные конкретного пациента
        p_measurements = df_measurements[df_measurements['id пациента'] == p_id].sort_values('timestamp')
        p_kzs = df_kzs_critical[df_kzs_critical['id пациента'] == p_id]
        p_cluster = df_clusters[df_clusters['patient_id'] == p_id]['cluster'].values[0]
        
        if len(p_measurements) < 5: # Пропускаем, если слишком мало данных
            continue
            
        start_date = p_measurements['date'].min()
        end_date = p_measurements['date'].max() - timedelta(days=horizon)
        
        current_date = start_date + timedelta(days=window_size)
        
        while current_date <= end_date:
            # Окно прошлого (Feature Window)
            history = p_measurements[(p_measurements['date'] >= current_date - timedelta(days=window_size)) & 
                                     (p_measurements['date'] < current_date)]
            
            # Окно будущего (Target Horizon)
            future_events = p_kzs[(p_kzs['kzs_date'] >= current_date) & 
                                  (p_kzs['kzs_date'] < current_date + timedelta(days=horizon))]
            
            if len(history) >= 3: # Минимум 3 измерения за 2 недели для прогноза
                sample = {
                    'patient_id': p_id,
                    'cluster': p_cluster,
                    'last_date': current_date,
                    # Базовые фичи (потом расширим)
                    'sbp_mean': history['САД'].mean(),
                    'sbp_std': history['САД'].std() if len(history) > 1 else 0,
                    'dbp_mean': history['ДАД'].mean(),
                    'hr_mean': history['ЧП'].mean(),
                    'meas_count': len(history),
                    # ЦЕЛЕВАЯ ПЕРЕМЕННАЯ
                    'target': 1 if len(future_events) > 0 else 0
                }
                samples.append(sample)
            
            # Сдвигаем на 3 дня, чтобы окна не дублировали друг друга слишком сильно
            current_date += timedelta(days=3)
            
    return pd.DataFrame(samples)


In [35]:
import pickle
import os

# ============================================
# СОЗДАНИЕ ОБУЧАЮЩЕЙ ВЫБОРКИ (с кешированием)
# ============================================

CACHE_FILE = os.path.join(CACHE_DIR, 'final_dataset.pkl')

if os.path.exists(CACHE_FILE):
    print(f"📦 Загружаем кешированную выборку из {CACHE_FILE}")
    final_dataset = pickle.load(open(CACHE_FILE, 'rb'))
    print(f"   Размер: {final_dataset.shape}")
else:
    print("🔄 Создаём обучающую выборку с нуля (это займёт ~15 минут)...")
    final_dataset = create_training_dataset(measurements, kzs, clusters_df)
    
    # Сохраняем
    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(final_dataset, f)
    print(f"✅ Сохранено в {CACHE_FILE}, размер: {final_dataset.shape}")

📦 Загружаем кешированную выборку из cache\final_dataset.pkl
   Размер: (404635, 9)


In [36]:
import pickle
import os
import numpy as np
import pandas as pd
from datetime import timedelta

# ============================================
# РАСЧЁТ TIME IN RANGE (TIR)
# ============================================

CACHE_TIR = os.path.join(CACHE_DIR, 'final_dataset_with_tir.pkl')

if os.path.exists(CACHE_TIR):
    print(f"📦 Загружаем кешированный датасет с TIR из {CACHE_TIR}")
    final_dataset = pickle.load(open(CACHE_TIR, 'rb'))
    print(f"   Размер: {final_dataset.shape}")
else:
    print("🔄 Рассчитываем Time in Range (TIR) для всех окон...")
    
    # Индексируем измерения для быстрого доступа
    measurements['date'] = pd.to_datetime(measurements['время измерения']).dt.date
    measurements_indexed = measurements.set_index(['id пациента', 'date']).sort_index()
    
    tir_values = []
    
    for p_id in tqdm(final_dataset['patient_id'].unique(), desc="TIR по пациентам"):
        p_windows = final_dataset[final_dataset['patient_id'] == p_id]
        
        if p_id not in measurements_indexed.index.get_level_values(0).unique():
            tir_values.extend([0] * len(p_windows))
            continue
            
        p_measurements = measurements_indexed.loc[p_id]
        
        for _, row in p_windows.iterrows():
            end_date = row['last_date']
            start_date = end_date - timedelta(days=14)
            
            # Берем срез данных за 14 дней
            window_data = p_measurements.loc[start_date:end_date - timedelta(days=1)] if start_date <= end_date else pd.DataFrame()
            
            if len(window_data) > 0:
                tir = (window_data['САД'] > 140).mean()
            else:
                tir = 0
            tir_values.append(tir)
    
    final_dataset['time_in_hypertension'] = tir_values
    
    # Добавляем статические признаки из кластеров
    static_features = clusters_df[['patient_id', 'возраст', 'ИМТ']].drop_duplicates()
    final_dataset = final_dataset.merge(static_features, on='patient_id', how='left')
    
    # Сохраняем
    with open(CACHE_TIR, 'wb') as f:
        pickle.dump(final_dataset, f)
    print(f"✅ Сохранено в {CACHE_TIR}")
    print(f"   Баланс классов: {final_dataset['target'].value_counts(normalize=True)}")

📦 Загружаем кешированный датасет с TIR из cache\final_dataset_with_tir.pkl
   Размер: (404635, 12)


In [38]:
import pickle
import os
import numpy as np
import pandas as pd
from datetime import timedelta
from tqdm import tqdm

# ============================================
# СОЗДАНИЕ 3D ТЕНЗОРА (ВРЕМЕННЫХ РЯДОВ)
# ============================================

CACHE_TENSOR = os.path.join(CACHE_DIR, 'X_tensor.pkl')
CACHE_Y = os.path.join(CACHE_DIR, 'y_tensor.pkl')
CACHE_SAMPLED = os.path.join(CACHE_DIR, 'df_sampled.pkl')

def create_3d_tensor_fast(df_measurements, final_df, sample_size=30000, window_size=14, max_points=20):
    """Быстрое создание 3D тензора для нейросети"""
    
    # Делаем выборку с балансировкой классов
    if len(final_df) > sample_size:
        df_kzs = final_df[final_df['target'] == 1]
        df_normal = final_df[final_df['target'] == 0].sample(sample_size - len(df_kzs), random_state=42)
        df_sampled = pd.concat([df_kzs, df_normal]).sample(frac=1, random_state=42).reset_index(drop=True)
    else:
        df_sampled = final_df.copy()
    
    # Группируем измерения для быстрого доступа
    df_measurements['timestamp'] = pd.to_datetime(df_measurements['время измерения'])
    df_measurements['date'] = df_measurements['timestamp'].dt.date
    measurements_dict = {p_id: group.sort_values('timestamp') 
                         for p_id, group in df_measurements.groupby('id пациента')}
    
    X_deep = []
    y_deep = []
    
    for _, row in tqdm(df_sampled.iterrows(), total=len(df_sampled), desc="Создание тензора"):
        p_id = row['patient_id']
        end_date = row['last_date']
        
        if p_id not in measurements_dict:
            X_deep.append(np.zeros((max_points, 3)))
            y_deep.append(row['target'])
            continue
            
        p_data = measurements_dict[p_id]
        
        # Фильтрация окна
        mask = (p_data['date'] >= end_date - timedelta(days=window_size)) & \
               (p_data['date'] < end_date)
        
        window_data = p_data[mask].tail(max_points)
        values = window_data[['САД', 'ДАД', 'ЧП']].values
        
        # Padding нулями в начало
        if len(values) < max_points:
            pad = np.zeros((max_points - len(values), 3))
            values = np.vstack([pad, values])
            
        X_deep.append(values)
        y_deep.append(row['target'])
    
    return np.array(X_deep), np.array(y_deep), df_sampled

if os.path.exists(CACHE_TENSOR) and os.path.exists(CACHE_SAMPLED):
    print(f"📦 Загружаем кешированный тензор...")
    X_tensor = pickle.load(open(CACHE_TENSOR, 'rb'))
    y_tensor = pickle.load(open(CACHE_Y, 'rb'))
    df_sampled = pickle.load(open(CACHE_SAMPLED, 'rb'))
    print(f"   X_tensor: {X_tensor.shape}")
    print(f"   Кризов в выборке: {sum(y_tensor)}")
else:
    print("🔄 Создаём 3D тензор для нейросети (это займёт ~3 минуты)...")
    X_tensor, y_tensor, df_sampled = create_3d_tensor_fast(measurements, final_dataset)
    
    with open(CACHE_TENSOR, 'wb') as f:
        pickle.dump(X_tensor, f)
    with open(CACHE_Y, 'wb') as f:
        pickle.dump(y_tensor, f)
    with open(CACHE_SAMPLED, 'wb') as f:
        pickle.dump(df_sampled, f)
    print(f"✅ Сохранено. X_tensor: {X_tensor.shape}, Кризов: {sum(y_tensor)}")

🔄 Создаём 3D тензор для нейросети (это займёт ~3 минуты)...


Создание тензора: 100%|██████████| 30000/30000 [00:47<00:00, 625.85it/s]


✅ Сохранено. X_tensor: (30000, 20, 3), Кризов: 16636


In [56]:
import pickle
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU, Dense, Dropout, Attention, GlobalAveragePooling1D
from sklearn.model_selection import train_test_split

# ============================================
# ОБУЧЕНИЕ НЕЙРОСЕТИ GRU + ATTENTION
# ============================================

CACHE_NN_PROBS = os.path.join(CACHE_DIR, 'nn_probs.pkl')
CACHE_NN_MODEL = os.path.join(CACHE_DIR, 'nn_model.h5')

def build_medical_model(input_shape):
    inputs = Input(shape=input_shape)
    
    # Нормализация в пределах модели
    x = tf.keras.layers.BatchNormalization()(inputs)
    
    # GRU слой
    gru_out = GRU(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)(x)
    
    # Механизм внимания
    query = Dense(64)(gru_out)
    value = Dense(64)(gru_out)
    attn_out = Attention()([query, value])
    
    # Классификация
    avg_pool = GlobalAveragePooling1D()(attn_out)
    drop = Dropout(0.3)(avg_pool)
    outputs = Dense(1, activation='sigmoid')(drop)
    
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

if os.path.exists(CACHE_NN_PROBS):
    print(f"📦 Загружаем кешированные вероятности нейросети...")
    nn_probs = pickle.load(open(CACHE_NN_PROBS, 'rb'))
    print(f"   Вероятности загружены, форма: {nn_probs.shape}")
else:
    print("🔄 Обучаем нейросеть (это займёт ~5 минут)...")
    
    # Нормализация входных данных
    X_scaled = X_tensor / 200.0
    
    # Разделение на train/val
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_tensor, test_size=0.2, random_state=42, stratify=y_tensor
    )
    
    # Обучение
    model_dl = build_medical_model((20, 3))
    history = model_dl.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=30, batch_size=64, verbose=1
    )
    
    # Сохраняем модель и вероятности для ВСЕХ данных
    model_dl.save(CACHE_NN_MODEL)
    nn_probs = model_dl.predict(X_scaled).flatten()
    
    with open(CACHE_NN_PROBS, 'wb') as f:
        pickle.dump(nn_probs, f)
    
    print(f"✅ Сохранено. Вероятности: {nn_probs.shape}")

# Убеждаемся, что nn_probs соответствует df_sampled
print(f"✅ nn_probs.shape = {nn_probs.shape}, df_sampled.shape = {df_sampled.shape}")

🔄 Обучаем нейросеть (это займёт ~5 минут)...
Epoch 1/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - AUC: 0.8344 - loss: 0.4967 - val_AUC: 0.8856 - val_loss: 0.4363
Epoch 2/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - AUC: 0.8643 - loss: 0.4593 - val_AUC: 0.8966 - val_loss: 0.4106
Epoch 3/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - AUC: 0.8782 - loss: 0.4398 - val_AUC: 0.9001 - val_loss: 0.4026
Epoch 4/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - AUC: 0.8844 - loss: 0.4289 - val_AUC: 0.9011 - val_loss: 0.3959
Epoch 5/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - AUC: 0.8903 - loss: 0.4173 - val_AUC: 0.9022 - val_loss: 0.3953
Epoch 6/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - AUC: 0.8921 - loss: 0.4138 - val_AUC: 0.9028 - val_loss: 0.3928
Epoch 7/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - AUC: 0.8928 - loss: 0.4128 - val_AUC: 0.9036 - val_loss: 0.3961
Epoch 8/30
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - AUC: 0.8962 - loss: 0.4066 - val_AUC: 0.9039 - val_loss: 0.3902
Epo

938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step
✅ Сохранено. Вероятности: (30000,)
✅ nn_probs.shape = (30000,), df_sampled.shape = (30000, 23)


In [57]:
import pickle
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

# ============================================
# XGBOOST НА СТАТИЧЕСКИХ ПРИЗНАКАХ
# ============================================

CACHE_XGB_PROBS = os.path.join(CACHE_DIR, 'xgb_probs.pkl')
CACHE_SPLIT = os.path.join(CACHE_DIR, 'train_test_split.pkl')

# Статические признаки для XGBoost
static_cols = ['sbp_mean', 'sbp_std', 'time_in_hypertension', 'cluster', 'возраст', 'ИМТ']

if os.path.exists(CACHE_XGB_PROBS) and os.path.exists(CACHE_SPLIT):
    print(f"📦 Загружаем кешированные вероятности XGBoost...")
    xgb_probs = pickle.load(open(CACHE_XGB_PROBS, 'rb'))
    split_data = pickle.load(open(CACHE_SPLIT, 'rb'))
    
    y_test = split_data['y_test']
    test_mask = split_data['test_mask']
    test_clusters = split_data['test_clusters']
    X_static_scaled = split_data['X_static_scaled']
    
    print(f"   XGBoost вероятности загружены, форма: {xgb_probs.shape}")
else:
    print("🔄 Обучаем XGBoost на статических признаках...")
    
    # Подготовка статических признаков
    X_static_raw = df_sampled[static_cols].fillna(0).values
    scaler = StandardScaler()
    X_static_scaled = scaler.fit_transform(X_static_raw)
    
    # Разделение по пациентам (важно для избежания утечки!)
    patient_ids = df_sampled['patient_id'].values
    unique_patients = np.unique(patient_ids)
    train_pat, test_pat = train_test_split(unique_patients, test_size=0.2, random_state=42)
    
    train_mask = np.isin(patient_ids, train_pat)
    test_mask = np.isin(patient_ids, test_pat)
    
    X_st_train = X_static_scaled[train_mask]
    X_st_test = X_static_scaled[test_mask]
    y_train = df_sampled['target'].values[train_mask]
    y_test = df_sampled['target'].values[test_mask]
    test_clusters = df_sampled['cluster'].values[test_mask]
    
    # Обучение XGBoost
    weight = (len(y_train) - sum(y_train)) / sum(y_train)
    xgb = XGBClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.05,
        scale_pos_weight=weight, random_state=42, eval_metric='logloss'
    )
    xgb.fit(X_st_train, y_train)
    xgb_probs = xgb.predict_proba(X_st_test)[:, 1]
    
    # Сохраняем всё необходимое
    with open(CACHE_XGB_PROBS, 'wb') as f:
        pickle.dump(xgb_probs, f)
    
    split_data = {
        'y_test': y_test,
        'test_mask': test_mask,
        'test_clusters': test_clusters,
        'X_static_scaled': X_static_scaled,
        'train_mask': train_mask
    }
    with open(CACHE_SPLIT, 'wb') as f:
        pickle.dump(split_data, f)
    
    print(f"✅ XGBoost обучен. Тестовая выборка: {len(y_test)} samples")
    print(f"   Кризов в тесте: {y_test.sum()} ({100*y_test.mean():.1f}%)")

# Берём нейросетевые вероятности для тестовых пациентов
nn_probs_test = nn_probs[test_mask]

print(f"\n📊 Итог:")
print(f"   nn_probs_test: {nn_probs_test.shape}")
print(f"   xgb_probs: {xgb_probs.shape}")
print(f"   y_test: {y_test.shape}")
print(f"   test_clusters: {test_clusters.shape}")

🔄 Обучаем XGBoost на статических признаках...
✅ XGBoost обучен. Тестовая выборка: 6132 samples
   Кризов в тесте: 3500 (57.1%)

📊 Итог:
   nn_probs_test: (6132,)
   xgb_probs: (6132,)
   y_test: (6132,)
   test_clusters: (6132,)


In [58]:
import numpy as np
from sklearn.metrics import f1_score, classification_report

# ============================================
# АДАПТИВНЫЙ БЛЕНДИНГ ПО КЛАСТЕРАМ
# ============================================

print("🚀 Адаптивный блендинг (Adaptive Blending) для сложных случаев...")

final_preds = np.zeros(len(y_test), dtype=int)
config_log = []

# Проходимся по каждому кластеру отдельно
for c_id in sorted(np.unique(test_clusters)):
    mask = (test_clusters == c_id)
    if mask.sum() < 30:
        continue

    # Вес для кластера (компенсация малого размера)
    cluster_weight = 1.0 + (500 / mask.sum())
    weights = np.ones(mask.sum()) * cluster_weight

    best_f1 = -1
    best_w = 0.73
    best_t = 0.5

    # Грубый поиск
    for w in np.linspace(0.1, 0.9, 9):
        for t in np.linspace(0.3, 0.7, 81):
            blend = w * nn_probs_test[mask] + (1 - w) * xgb_probs[mask]
            preds = (blend > t).astype(int)
            f1 = f1_score(y_test[mask], preds, average='macro', 
                          sample_weight=weights, zero_division=0)
            if f1 > best_f1:
                best_f1, best_w, best_t = f1, w, t

    # Тонкий поиск
    for w in np.linspace(max(0.1, best_w - 0.1), min(0.9, best_w + 0.1), 21):
        for t in np.linspace(max(0.2, best_t - 0.05), min(0.8, best_t + 0.05), 101):
            blend = w * nn_probs_test[mask] + (1 - w) * xgb_probs[mask]
            preds = (blend > t).astype(int)
            f1 = f1_score(y_test[mask], preds, average='macro',
                          sample_weight=weights, zero_division=0)
            if f1 > best_f1:
                best_f1, best_w, best_t = f1, w, t

    config_log.append((c_id, best_w, best_t, best_f1))
    blend = best_w * nn_probs_test[mask] + (1 - best_w) * xgb_probs[mask]
    final_preds[mask] = (blend > best_t).astype(int)

# Вывод результатов
print("\n📋 Оптимальные настройки по кластерам:")
for c_id, w, t, f1 in config_log:
    print(f"  ✅ Кластер {c_id}: W_nn={w:.2f}, Thresh={t:.3f}, F1={f1:.3f} (n={np.sum(test_clusters==c_id)})")

# Итоговая оценка
final_f1 = f1_score(y_test, final_preds, average='macro')
print(f"\n🏆 ИТОГОВЫЙ MACRO F1 (Adaptive Blending): {final_f1:.4f}")

# Детальный отчёт
print("\n📋 Classification Report:")
print(classification_report(y_test, final_preds, target_names=['Стабильные', 'Криз']))


🚀 Адаптивный блендинг (Adaptive Blending) для сложных случаев...

📋 Оптимальные настройки по кластерам:
  ✅ Кластер 1: W_nn=0.41, Thresh=0.582, F1=0.829 (n=1120)
  ✅ Кластер 2: W_nn=0.62, Thresh=0.438, F1=0.856 (n=2324)
  ✅ Кластер 3: W_nn=0.89, Thresh=0.347, F1=0.847 (n=2688)

🏆 ИТОГОВЫЙ MACRO F1 (Adaptive Blending): 0.8512

📋 Classification Report:
              precision    recall  f1-score   support

  Стабильные       0.83      0.83      0.83      2632
        Криз       0.87      0.87      0.87      3500

    accuracy                           0.85      6132
   macro avg       0.85      0.85      0.85      6132
weighted avg       0.85      0.85      0.85      6132

